# 04 · Exportação da Gold para CSV
### V-Energy Grid 360 — Arquitetura Medalhão


Exporta as tabelas da camada Gold para **CSV**, num Volume, e escreve um `manifesto.json` com as contagens e os tipos de cada coluna. Os CSVs e o manifesto são a entrada do script `carregar_sqlite.py`, que monta o banco SQLite local do portal e do agente de IA (o case pede: "exportar a Gold em CSV e popular um banco local").

**Fluxo:**
1. **Aqui (Databricks):** `gold.<tabela>` → `/Volumes/v_energy/gold/export/<tabela>/part-*.csv` + `manifesto.json`.
2. **Caminho curto (seção 6):** o próprio notebook roda `carregar_sqlite.py` no Databricks e deixa **um único arquivo**, `gold.db.zip`, no Volume. Basta baixar esse arquivo.
3. **Caminho manual:** baixar a pasta `export` para `dados_gold/` e rodar `python carregar_sqlite.py` na sua máquina.


**Decisões:**
| Decisão | Por quê |
|---|---|
| Nulo vira **campo vazio** (`nullValue=""`) | o carregador converte vazio em `NULL` de verdade; sem isso o CSV traria a palavra `null`, que no SQLite viraria texto no meio de colunas numéricas |
| Datas em `yyyy-MM-dd` e horários em `yyyy-MM-dd HH:mm:ss`, **sem fuso** | o mesmo formato das Silvers; o SQLite guarda como texto ISO |
| Tabelas grandes em **vários arquivos** (`repartition`) | o download pela interface do Databricks pode ter limite de tamanho (não verificado); a célula final imprime o tamanho real de cada arquivo |
| `manifesto.json` com linhas e tipos | o carregador cria as tabelas com tipos explícitos e confere as contagens: um CSV truncado no download é detectado antes de chegar ao portal |
| Releitura dos CSVs e comparação com a tabela | prova que o que foi escrito tem as mesmas linhas e colunas |

**Pré-requisito:** as Golds listadas em `TABELAS` já gravadas.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

import json
from pyspark.sql import functions as F

volume = f"{catalog}.gold.export"
base = f"/Volumes/{catalog}/gold/export"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume}")
print("destino:", base)

## 2. Tabelas a exportar

O número é a quantidade de arquivos CSV em que a tabela é dividida. Tabelas pequenas, com **1** arquivo. A `fato_consumo_faturamento` tem 2,15 milhões de linhas e 31 colunas, e o tamanho do CSV não foi medido: ela começa com **6** arquivos, e a seção 5 mostra o tamanho real para ajustar.

Acrescente as demais Golds aqui conforme ficarem prontas. Se o nome estiver na lista e a tabela ainda não existir, o notebook para com um erro claro.

In [0]:
TABELAS = {
    "dim_uc": 1,
    "fato_consumo_faturamento": 6,
    "fato_atendimento": 2,
    "receita_decomposicao": 1,
    "fato_segmento_mensal": 1,
    "kpis_mensal": 1,
    "fato_troca_medidor": 1,
    "fato_interrupcao": 1,
    "dec_fec_alimentador_mensal": 1,
    "fila_inspecao": 1,
}
print("tabelas:", list(TABELAS))

## 3. Exportação e releitura

Para cada tabela:
1. lê a Gold e conta as linhas;
2. grava os CSVs (`header`, nulo vazio, datas ISO);
3. **relê** os CSVs e compara linhas e nomes de colunas com a tabela de origem;
4. guarda no manifesto as linhas e os tipos de cada coluna (`df.dtypes`).

Se a releitura não bater, o notebook para e **não escreve o manifesto**, então um carregamento com dados incompletos não acontece sem aviso.

A releitura descarta o cabeçalho de cada arquivo (é o comportamento do Spark com `header=true`), por isso a contagem tem que ser igual à da tabela.

In [0]:
manifesto = {}

for nome, n_arquivos in TABELAS.items():
    df = spark.table(f"{gold_schema}.{nome}")            # erro claro se a tabela não existir
    n = df.count()
    destino = f"{base}/{nome}"

    (
        df.repartition(n_arquivos)
        .write.mode("overwrite")
        .option("header", True)
        .option("nullValue", "")
        .option("dateFormat", "yyyy-MM-dd")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .csv(destino)
    )

    lido = spark.read.option("header", True).csv(destino)
    n_lido = lido.count()
    if n_lido != n or lido.columns != df.columns:
        raise Exception(f"{nome}: releitura não bate (tabela {n} linhas, CSV {n_lido}; "
                        f"colunas iguais: {lido.columns == df.columns})")

    manifesto[nome] = {"linhas": n, "colunas": [{"nome": c, "tipo": t} for c, t in df.dtypes]}
    print(f"[OK] {nome}: {n:,} linhas, {len(df.columns)} colunas, {n_arquivos} arquivo(s)".replace(",", "."))

## 4. Manifesto

Um arquivo pequeno com, para cada tabela, o número de linhas e o nome e o tipo (do Spark) de cada coluna. O carregador usa os tipos para criar as tabelas do SQLite e as linhas para conferir a carga.

In [0]:
with open(f"{base}/manifesto.json", "w", encoding="utf-8") as f:
    json.dump(manifesto, f, ensure_ascii=False, indent=1)

for nome, info in manifesto.items():
    print(nome, info["linhas"], "linhas")
print("manifesto gravado em", f"{base}/manifesto.json")

## 5. O que baixar

Lista os arquivos e os tamanhos reais. **É a medição que faltava** para saber se o download pela interface funciona e quantos arquivos a `fato_consumo_faturamento` precisa. Se algum arquivo ficar grande demais para baixar, aumente o número dele em `TABELAS` e rode de novo.

No Databricks: **Catalog → v_energy → gold → Volumes → export**. Baixe `manifesto.json` e os arquivos `part-*.csv` de cada tabela (os arquivos que começam com `_` são de controle do Spark e podem ficar). Monte esta pasta na sua máquina:

```
dados_gold/
  manifesto.json
  dim_uc/part-....csv
  fato_consumo_faturamento/part-....csv   (todos os arquivos)
```

Depois: `python data-engineering/scripts/carregar_sqlite.py --entrada dados_gold --saida gold.db`.

In [0]:
for nome in TABELAS:
    arquivos = [f for f in dbutils.fs.ls(f"{base}/{nome}") if f.name.endswith(".csv")]
    total = sum(f.size for f in arquivos) / 1e6
    maior = max(f.size for f in arquivos) / 1e6
    print(f"{nome}: {len(arquivos)} arquivo(s), {total:,.1f} MB no total, maior arquivo {maior:,.1f} MB".replace(",", "X").replace(".", ",").replace("X", "."))

## 6. Montar o `gold.db` aqui mesmo (caminho curto)

Em vez de baixar um CSV por vez, o notebook roda o `carregar_sqlite.py` **no Databricks**, lendo os CSVs que acabou de gravar, e deixa um único arquivo para baixar: `gold.db.zip`, no mesmo Volume. O carregador usa só a biblioteca padrão do Python, e as conferências (contagens, chaves e números do case) aparecem na saída desta célula.

O script é encontrado sozinho: o notebook procura o carregar_sqlite.py em data-engineering/scripts/ (no próprio repositório) e, se não achar, no Volume. Nenhum passo manual é necessário, nem no Workflow.

**Como funciona:**
1. o banco é montado em `/tmp` (disco local do ambiente; o SQLite precisa de escrita com salto, que o Volume não oferece);
2. o `gold.db` é compactado (`.zip`) em `/tmp` e **só então** copiado para o Volume (escrita sequencial);
3. se o carregador falhar em qualquer conferência, o notebook para com erro e **não cria o `.zip`**.

**Não verificado:** este caminho foi testado fora do Databricks, com dados de teste. A leitura dos CSVs pelo caminho `/Volumes/...` e a escrita em `/tmp` são o comportamento esperado do ambiente serverless, mas a primeira execução real é a confirmação. Se alguma delas falhar, o caminho manual continua valendo.

In [0]:
import os, shutil, subprocess, sys, zipfile

# O carregador fica no repositório (data-engineering/scripts/). Procura primeiro ali, que funciona
# em qualquer workspace e no Workflow; a cópia no Volume fica só como alternativa.
candidatos = [
    os.path.abspath(os.path.join(os.getcwd(), "..", "scripts", "carregar_sqlite.py")),  # no repositório (Git folder ou Job lendo do Git)
    f"{base}/carregar_sqlite.py",                                                      # cópia no Volume (caminho antigo)
]
SCRIPT = next((p for p in candidatos if os.path.exists(p)), None)
if SCRIPT is None:
    raise Exception(f"carregar_sqlite.py não encontrado em: {candidatos}")
print("script:", SCRIPT)

DB_LOCAL = "/tmp/gold.db"
ZIP_LOCAL = "/tmp/gold.db.zip"
ZIP_VOLUME = f"{base}/gold.db.zip"

print(f"espaço livre em /tmp: {shutil.disk_usage('/tmp').free / 1e9:,.1f} GB")

r = subprocess.run([sys.executable, SCRIPT, "--entrada", base, "--saida", DB_LOCAL], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise Exception("carregar_sqlite.py falhou (veja a saída acima). O gold.db.zip anterior, se existir, não foi alterado.")

## 7. Compactar e deixar o `gold.db.zip` no Volume

O banco é compactado em `/tmp` e copiado para o Volume. Esta célula imprime os tamanhos do `gold.db` e do `.zip` (a medição que faltava) e confere as contagens do arquivo compactado, abrindo o banco extraído em modo leitura.

**Download:** Catalog → v_energy → gold → Volumes → export → `gold.db.zip`. Descompacte e entregue o `gold.db` ao Dev (`backend/data/gold.db`).

In [0]:
with zipfile.ZipFile(ZIP_LOCAL, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    z.write(DB_LOCAL, arcname="gold.db")
shutil.copyfile(ZIP_LOCAL, ZIP_VOLUME)
mb = lambda p: os.path.getsize(p) / 1e6
print(f"gold.db: {mb(DB_LOCAL):,.1f} MB | gold.db.zip: {mb(ZIP_VOLUME):,.1f} MB (em {ZIP_VOLUME})")

# conferência do arquivo entregue: extrai uma cópia e conta as linhas de cada tabela
with zipfile.ZipFile(ZIP_VOLUME) as z:
    z.extract("gold.db", "/tmp/conferencia_zip")
import sqlite3
con = sqlite3.connect("file:/tmp/conferencia_zip/gold.db?mode=ro", uri=True)
for (tabela,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"):
    n = con.execute(f'SELECT COUNT(*) FROM "{tabela}"').fetchone()[0]
    print(f"  {tabela}: {n:,}".replace(",", "."))
con.close()
